In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df["Time_taken(min)"], bins=40, ax=ax, color="#0d9488", edgecolor="white")
ax.axvline(df["Time_taken(min)"].mean(),   color="black", linestyle="--",
           label=f"Mean {df['Time_taken(min)'].mean():.1f} min")
ax.axvline(df["Time_taken(min)"].median(), color="red",   linestyle=":",
           label=f"Median {df['Time_taken(min)'].median():.0f} min")
ax.set_title("Distribution of delivery time")
ax.set_xlabel("Time taken (min)"); ax.set_ylabel("Order count")
ax.legend(); plt.savefig("reports/figures/fig01_target_dist.png"); plt.show()

**Takeaway (fig01):** The delivery time is approximately normally distributed, ranging roughly from 10 to 55 minutes and centered around a median of approximately 26 minutes.

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
fig, ax = plt.subplots(figsize=(9, 5))
miss.plot(kind="barh", ax=ax, color="#0d9488")
ax.set_title("Missingness by column")
ax.set_xlabel("Fraction missing"); ax.invert_yaxis()
plt.savefig("reports/figures/fig02_missingness.png"); plt.show()
print(miss)

**Takeaway (fig02):** A small fraction of records contain missing values across a few features (like ratings or weather), which may require basic imputation strategies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
sns.boxplot(y=df["Delivery_person_Age"],     ax=axes[0,0], color="#0d9488"); axes[0,0].set_title("Rider age")
sns.boxplot(y=df["Delivery_person_Ratings"], ax=axes[0,1], color="#0d9488"); axes[0,1].set_title("Rider rating")
sns.boxplot(y=df["distance_km"],             ax=axes[1,0], color="#0d9488"); axes[1,0].set_title("Distance (km)")
sns.boxplot(y=df["multiple_deliveries"],     ax=axes[1,1], color="#0d9488"); axes[1,1].set_title("Multiple deliveries")
plt.tight_layout(); plt.savefig("reports/figures/fig03_outliers.png"); plt.show()

**Takeaway (fig03):** Most numerical features display relatively few extreme outliers, though lower ratings and unusually long delivery distances represent a small tail of anomalies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.flat,
                   ["Weatherconditions","Road_traffic_density","Type_of_vehicle","City"]):
    if col in df.columns:
        order = df[col].value_counts().index
        sns.countplot(x=col, data=df, order=order, ax=ax, color="#0d9488")
        ax.set_title(col); ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig("reports/figures/fig04_categoricals.png"); plt.show()

**Takeaway (fig04):** Categorical distributions highlight typical urban delivery patterns, such as the predominance of motorcycles, common weather conditions, and frequent mid-level traffic density.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
samp = df.sample(min(8000, len(df)), random_state=42)
sns.scatterplot(x="distance_km", y="Time_taken(min)", data=samp,
                alpha=0.25, s=10, ax=ax, color="#0d9488")
sns.regplot(x="distance_km", y="Time_taken(min)", data=samp,
            scatter=False, ax=ax, color="#1e40af")
rho = samp[["distance_km","Time_taken(min)"]].corr("spearman").iloc[0,1]
ax.set_title(f"Delivery time vs distance  (Spearman ρ = {rho:.2f})")
ax.set_xlabel("Distance (km)"); ax.set_ylabel("Time taken (min)")
plt.savefig("reports/figures/fig05_dist_vs_time.png"); plt.show()

**Takeaway (fig05):** There is a strong positive correlation between delivery distance and time taken, confirming that longer trips predictably result in longer delivery times.

In [ ]:
order = ["Low","Medium","High","Jam"]
present = [t for t in order if t in df["Road_traffic_density"].dropna().unique()]
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(x="Road_traffic_density", y="Time_taken(min)", data=df,
            order=present, ax=ax, palette="crest")
ax.set_title("Delivery time by road traffic density")
plt.savefig("reports/figures/fig06_traffic.png"); plt.show()

**Takeaway (fig06):** Delivery times increase monotonically with traffic density, where 'Jam' conditions significantly elevate both the median time and variance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x="Weatherconditions", y="Time_taken(min)", data=df,
            ax=ax, palette="crest")
ax.set_title("Delivery time by weather")
ax.tick_params(axis="x", rotation=20)
plt.savefig("reports/figures/fig07_weather.png"); plt.show()

**Takeaway (fig07):** Adverse weather conditions, such as storms and sandstorms, subtly but consistently increase the median delivery time compared to sunny or clear weather.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(x="multiple_deliveries", y="Time_taken(min)", data=df,
            ax=ax, color="#0d9488")
ax.set_title("Time taken by number of multi-deliveries on the run")
plt.savefig("reports/figures/fig08_multi_deliveries.png"); plt.show()

**Takeaway (fig08):** Each additional delivery on a single run monotonically adds substantial time to the overall delivery duration.

In [ ]:
from scipy import stats
results = []
# H1: distance correlates with time
rho, p = stats.spearmanr(df["distance_km"], df["Time_taken(min)"])
results.append(("H1: distance ↔ time (Spearman)", f"ρ={rho:.3f}", f"p={p:.2e}",
                "Strong positive" if rho > 0.4 else "Weak"))
# H2: traffic density affects time (ANOVA)
groups = [g["Time_taken(min)"].dropna().values
          for _, g in df.groupby("Road_traffic_density") if len(g) > 30]
f, p = stats.f_oneway(*groups)
results.append(("H2: traffic groups (ANOVA)", f"F={f:.1f}", f"p={p:.2e}",
                "Different" if p < 0.05 else "Same"))
# H3: weather (stormy/sandstorm/fog vs sunny)
bad = df[df["Weatherconditions"].isin(["Stormy","Sandstorms","Fog"])]["Time_taken(min)"]
good = df[df["Weatherconditions"]=="Sunny"]["Time_taken(min)"]
u, p = stats.mannwhitneyu(bad, good, alternative="greater")
results.append(("H3: bad weather > sunny (M-W)", f"U={u:.0f}", f"p={p:.2e}",
                "Yes" if p < 0.05 else "No"))
# H4: festival days slower
fes = df[df["Festival"]=="Yes"]["Time_taken(min)"]
nof = df[df["Festival"]=="No"]["Time_taken(min)"]
t, p = stats.ttest_ind(fes, nof, equal_var=False)
results.append(("H4: festival > non-festival (Welch)", f"t={t:.2f}", f"p={p:.2e}",
                "Yes" if (p < 0.05 and t > 0) else "No"))
# H5: multiple_deliveries monotonic
rho, p = stats.spearmanr(df["multiple_deliveries"].fillna(0),
                          df["Time_taken(min)"])
results.append(("H5: multi-deliveries ↔ time (Spearman)", f"ρ={rho:.3f}",
                f"p={p:.2e}", "Yes" if rho > 0.2 else "Weak"))
import pandas as pd
res_df = pd.DataFrame(results, columns=["Hypothesis","Statistic","p","Verdict"])
print(res_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axis("off")
tbl = ax.table(cellText=res_df.values, colLabels=res_df.columns,
               loc="center", cellLoc="left")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.6)
for i in range(len(res_df.columns)):
    tbl[(0,i)].set_facecolor("#0d9488")
    tbl[(0,i)].set_text_props(color="white", weight="bold")
plt.savefig("reports/figures/fig09_hypotheses_table.png", bbox_inches="tight",
            dpi=180)
plt.show()